In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# 시드 고정 (재현성)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
X_clf, y_clf = make_classification(
    n_samples=400, n_features=2, random_state=42,
    n_informative=2, n_redundant=0,n_clusters_per_class=1,class_sep=1.5
    )
scaler = StandardScaler()
# 데이터 분할 train - test 8:2
x_train,x_test,y_train,y_test = train_test_split(X_clf,y_clf,random_state=42, stratify=y_clf,test_size=0.2)
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
# pytorch 텐서로 변환
X_clf_train_tensor = torch.tensor(x_train,dtype=torch.float32)
y_clf_train_tensor = torch.FloatTensor(y_train)
x_test_t = torch.FloatTensor(x_test)
y_test_t = torch.FloatTensor(y_test)

In [ ]:
# 분류모델 정의
class BinaryClassifier(nn.Module):
  def __init__(self,input_dim, hidden_dim) -> None:
    super().__init__()
    self.fc1 = nn.Linear( input_dim , hidden_dim)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(hidden_dim ,1 )
  def forward(self, x):
    x = self.relu( self.fc1(x) )
    output = self.fc2(x)
    return output
model_clf = BinaryClassifier(2,16)
model_clf

In [ ]:
from tqdm import tqdm
model_clf = BinaryClassifier(2,16)
criterion = nn.BCEWithLogitsLoss()
optimizer_momentum = optim.SGD(model_clf.parameters(), lr=0.01, momentum=0.9)

x_train_dataset = TensorDataset(x_train_t,y_train_t)
x_train_loader = DataLoader(x_train_dataset,batch_size=16,shuffle=True)
# 훈련루프
epochs = 100
epoch_losses = []
for epoch in tqdm(range(epochs)):
  epoch_loss = 0
  for data, label in x_train_loader:
    optimizer_momentum.zero_grad();
    predict = model_clf(data).squeeze(1)
    loss = criterion(predict, label)
    epoch_loss +=  loss.item()
  epoch_loss / len(x_train_loader)
  epoch_losses.append(epoch_loss)

# loss값들을 저장해서 시각화 vs sgd
plt.plot(range(1,epochs+1),epoch_losses )
epoch_losses
